# Package seg-model v1 for fast prediction (M4-T01)

The selected model (nnU-Net, `models/registry/seg-model/v1`) is exported once to **TorchScript**. After that, one call
turns a scan into its tumor mask, without the 250 MB training checkpoint or the nnU-Net results folder:

```python
from bmts.deploy.segmenter import FastSegmenter
seg, timings = FastSegmenter.from_dir(EXPORT_DIR).segment_to_file(image_files, "case_seg.nii.gz")
```

This notebook:
1. checks the checkpoint is the registered seg-model v1 (SHA-256) and exports it to `BrainMRI_Data/models/seg-model-v1-torchscript/`;
2. on `N_CASES` **validation** patients, compares the exported model with the original nnU-Net prediction
   (what `nnUNetv2_predict` gives), with the 8 mirror flips (`tta=True`) and without them (`tta=False`, about 8x faster);
3. measures the time per scan and writes `per_case.csv` and `summary.json` to `BrainMRI_Data/runs/M4-T01_export/`.

Locked test patients are never loaded.

**How to run:** **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About 20–30 minutes for 20 patients. Run all again after a stop: a finished export is reused.

> Research prototype for decision support, not a diagnostic device.

In [ ]:
# 1. Settings, GPU and Drive
BRANCH = "main"
N_CASES = 20          # validation patients used for the comparison (the first N by ID)
DEVICE = "cuda"

import os, sys, json, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
CLEANED_TAR = BASE / "processed/brats2021_trainval_v1.tar"
SPLIT_CSV = BASE / "qc/split_v1.csv"
MODEL_DIR = BASE / "runs/M2-T04_nnunet/nnUNet_results/Dataset501_BraTS2021/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres"
EXPORT_DIR = BASE / "models/seg-model-v1-torchscript"
OUT = BASE / "runs/M4-T01_export"
OUT.mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: change the runtime type to T4 GPU"

In [ ]:
# 2. Code from the team repo, and the packages (nnU-Net version from the model manifest)
if Path("/content/repo/.git").exists():
    !git -C /content/repo fetch -q origin && git -C /content/repo checkout -q {BRANCH} && git -C /content/repo pull -q origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/nhahub/NHA-5-187.git /content/repo
!git -C /content/repo log --oneline -1
!pip -q install "nnunetv2==2.8.1"
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):   # nnU-Net warns when these are unset
    os.environ.setdefault(k, "/content/nnunet_unused")
sys.path.insert(0, "/content/repo/src")
import torch
print("torch", torch.__version__)

In [ ]:
# 3. Does the repo's copy of plans.json / dataset.json (used only by the tests) match the trained model's?
#    The export always copies the model's own files, so a difference here only means the repo copy needs updating.
def differences(a, b, path=""):
    if isinstance(a, dict) and isinstance(b, dict):
        return [d for k in sorted(set(a) | set(b)) for d in differences(a.get(k), b.get(k), f"{path}/{k}")]
    if isinstance(a, list) and isinstance(b, list) and len(a) == len(b):
        return [d for i, (x, y) in enumerate(zip(a, b)) for d in differences(x, y, f"{path}[{i}]")]
    if isinstance(a, float) and isinstance(b, float):
        return [] if abs(a - b) <= 1e-9 * max(1.0, abs(b)) else [path]
    return [] if a == b else [path]

for name in ("plans.json", "dataset.json"):
    diff = differences(json.load(open(f"/content/repo/models/registry/seg-model/v1/nnunet/{name}")),
                       json.load(open(MODEL_DIR / name)))
    print(name, "matches" if not diff else f"DIFFERS at {diff[:10]}: copy the model's file into the repo")

In [ ]:
# 4. Export (the script first checks the checkpoint SHA-256 against models/registry/seg-model/v1/manifest.json).
#    Skipped when a complete export of the registered checkpoint is already on Drive.
from bmts.deploy.export import sha256
%cd /content/repo
registered = json.load(open("/content/repo/models/registry/seg-model/v1/manifest.json"))["checkpoint_file"]["sha256"]
done = False
if (EXPORT_DIR / "deploy_manifest.json").exists():
    m = json.load(open(EXPORT_DIR / "deploy_manifest.json"))
    done = (m["source"]["checkpoint_sha256"] == registered and (EXPORT_DIR / m["model_file"]).exists()
            and sha256(EXPORT_DIR / m["model_file"]) == m["model_sha256"])
if done:
    print("export already on Drive, from the registered checkpoint:", m["exported_utc"])
else:
    !python scripts/export_model.py --model-dir "{MODEL_DIR}" --out "{EXPORT_DIR}"
!ls -la "{EXPORT_DIR}"

In [ ]:
# 5. The first N_CASES validation patients, restored from the cleaned archive (locked patients are refused)
import pandas as pd
from bmts.common.constants import MODALITIES, SEG
from bmts.data.splits import check_subject       # light split check (pandas only), no training loader needed
split = pd.read_csv(SPLIT_CSV, dtype=str, keep_default_na=False)
val = split[(split.dataset_source == "brats2021") & (split.split == "val")].canonical_subject_id
subjects = sorted(s.split("__", 1)[-1] for s in val.unique())[:N_CASES]
assert all(check_subject(SPLIT_CSV, "brats2021", s) == "val" for s in subjects)   # raises for locked patients
members = " ".join(f"brats2021/{s}" for s in subjects)
!mkdir -p /content/processed && tar -xf "{CLEANED_TAR}" -C /content/processed {members}
cases = []
for s in subjects:
    for tp in sorted(p for p in (Path("/content/processed/brats2021") / s).iterdir() if p.is_dir()):
        case = {"id": s, **{m: tp / f"{m}.nii.gz" for m in (*MODALITIES, SEG)}}
        assert all(Path(case[k]).is_file() for k in (*MODALITIES, SEG)), f"missing files in {tp}"
        cases.append(case)
print(len(cases), "validation scans:", [c["id"] for c in cases][:5], "...")

In [ ]:
# 6. Original nnU-Net prediction vs the exported model (with and without the 8 mirror flips)
import numpy as np
from nnunetv2.inference.export_prediction import convert_predicted_logits_to_segmentation_with_correct_shape
from bmts.common.constants import MODALITIES, REGIONS
from bmts.deploy.segmenter import FastSegmenter
from bmts.uncertainty import nnunet_tta as T

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def original(predictor, files):
    """What nnUNetv2_predict does with seg-model v1 (8 mirror flips), timed like FastSegmenter.segment."""
    sync(); t0 = time.perf_counter()
    data, props = T.preprocess(predictor, files)
    logits = predictor.predict_logits_from_preprocessed_data(torch.from_numpy(data))
    seg = convert_predicted_logits_to_segmentation_with_correct_shape(
        logits, predictor.plans_manager, predictor.configuration_manager, predictor.label_manager, props)
    sync()
    return np.asarray(seg, np.uint8), time.perf_counter() - t0

def dice(a, b):
    s = a.sum() + b.sum()
    return 1.0 if s == 0 else 2 * (a & b).sum() / s

t0 = time.perf_counter()
reference = T.load_predictor(MODEL_DIR, device=DEVICE)
reference.use_mirroring = True                       # nnUNetv2_predict default
load_original_s = time.perf_counter() - t0
t0 = time.perf_counter()
fast = {tta: FastSegmenter.from_dir(EXPORT_DIR, device=DEVICE, tta=tta) for tta in (True, False)}
load_export_s = (time.perf_counter() - t0) / 2

rows = []
for i, case in enumerate(cases):
    files = [case[m] for m in MODALITIES]
    expert = T.read_sitk(case["seg"]).astype(np.uint8)
    ref_seg, ref_s = original(reference, files)
    row = {"case": case["id"], "original_s": ref_s}
    for tta, segmenter in fast.items():
        sync()
        seg, _, timings = segmenter.segment(files)
        sync()
        tag = "tta" if tta else "fast"
        row[f"{tag}_s"] = timings["total_s"]
        row[f"{tag}_network_s"] = timings["network_s"]
        row[f"{tag}_voxels_differ"] = int((seg != ref_seg).sum())
        for region, labels in REGIONS.items():
            row[f"{tag}_dice_vs_original_{region}"] = dice(np.isin(seg, labels), np.isin(ref_seg, labels))
            row[f"{tag}_dice_expert_{region}"] = dice(np.isin(seg, labels), np.isin(expert, labels))
    for region, labels in REGIONS.items():
        row[f"original_dice_expert_{region}"] = dice(np.isin(ref_seg, labels), np.isin(expert, labels))
    rows.append(row)
    print(f"{i + 1}/{len(cases)} {case['id']}: original {ref_s:.1f} s | tta {row['tta_s']:.1f} s "
          f"({row['tta_voxels_differ']} voxels differ) | fast {row['fast_s']:.1f} s ({row['fast_voxels_differ']} differ)",
          flush=True)
per_case = pd.DataFrame(rows)
per_case.to_csv(OUT / "per_case.csv", index=False)

In [ ]:
# 7. Summary (the first case is left out of the times: it includes GPU warm-up)
timed = per_case.iloc[1:] if len(per_case) > 1 else per_case
deploy_manifest = json.load(open(EXPORT_DIR / "deploy_manifest.json"))
summary = {
    "cases": len(per_case), "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
    "torch": torch.__version__, "export": {k: deploy_manifest[k] for k in ("model_sha256", "export_check_max_abs_diff")},
    "checkpoint_sha256": deploy_manifest["source"]["checkpoint_sha256"],
    "load_s": {"original": load_original_s, "export": load_export_s},
    "median_s_per_scan": {k: float(timed[f"{k}_s"].median()) for k in ("original", "tta", "fast")},
    "voxels_differ_from_original": {k: {"median": float(per_case[f"{k}_voxels_differ"].median()),
                                        "max": int(per_case[f"{k}_voxels_differ"].max())} for k in ("tta", "fast")},
    "min_dice_vs_original": {k: {r: float(per_case[f"{k}_dice_vs_original_{r}"].min()) for r in REGIONS}
                             for k in ("tta", "fast")},
    "mean_dice_expert": {k: {r: float(per_case[f"{k}_dice_expert_{r}"].mean()) for r in REGIONS}
                         for k in ("original", "tta", "fast")},
}
json.dump(summary, open(OUT / "summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# 8. Finish writing to Drive
drive.flush_and_unmount()